In [1]:
## Standard libraries
import os
import numpy as np
from collections import Counter

## PyTorch & Torchvision
import torch
import torchvision
from torchvision import transforms
from torchinfo import summary
import torchvision.models as models
from torch.utils.data import DataLoader

# Classification metrics
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

## Imports for plotting
import matplotlib.pyplot as plt
import matplotlib.cm as cm

## tqdm for loading bars
from tqdm import tqdm

In [2]:
# Decide which device we want to run on
device = (
    "cuda"
    if  (torch.cuda.is_available())
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

#Additional Info when using cuda
if device == 'cuda':
    print('Number of GPU(s):', torch.cuda.device_count())
    print('GPU device(s):', torch.cuda.get_device_name(0))

Using cuda device
Number of GPU(s): 1
GPU device(s): NVIDIA GeForce GTX 1650 with Max-Q Design


In [6]:
# Path to the data and the weigth (to be saved) directories 
dataset_dir = ""
model_weigths_path = ""

# Batch size to be utilized
batch_size = 32

# Rate to reduce the database size
reduce_rate = 40

In [8]:
def get_data_loaders(dataset_dir, batch_size=16, reduce_rate=10):

    # Define the transforms applied to the inputed data
    fashion_mnist = torchvision.datasets.FashionMNIST(download=True, train=True, root=dataset_dir).train_data.float()

    
    data_transform = transforms.Compose([ transforms.Resize((224, 224)),
                                         #transforms.RandomHorizontalFlip(0.5),
                                         #transforms.RandomResizedCrop(224),
                                         transforms.ToTensor(), 
                                         transforms.Normalize((fashion_mnist.mean()/255,), (fashion_mnist.std()/255,))])

   
    # Load the data realted to the training set
    data = torchvision.datasets.FashionMNIST(download=True, root=dataset_dir, transform=data_transform, train=False)

    # Get the names of the classes
    class_names = data.classes
    
    print("class names and associated index")
    print(data.class_to_idx)

    # Get the number of channels
    num_channels  = data[0][0].shape[0]
    print(f"Image shape: {data[0][0].shape}")
    
    # Reduce the amount of data -> reduce the memory size and the computational time
    torch.manual_seed(0) # to select the same data at each run
    data_limit_len = round(len(data) /reduce_rate)

    # Split into training and validation sets - with limited number of images
    #test_dataset = torch.utils.data.dataset.random_split(data, )
    test_dataset = data
    # Check how many images per class - time consuming !
    test_classes = [label for _, label in test_dataset]
    print("Number of images per class in the train set:")
    print(sorted(Counter(test_classes).items()))

    # Create the data loader for the training and validation sets
    test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    # shuffle=True -> this reshuffles the data at every epoch
    # drop_last=True -> remove the last batch if is not full
        
    return test_loader, class_names, num_channels

In [9]:
# load the data
test_loader, classes_names, num_channels = get_data_loaders(dataset_dir, batch_size, reduce_rate)

print("The test set contains {} images, in {} batches".format(len(test_loader.dataset), len(test_loader)))

class names and associated index
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}
Image shape: torch.Size([1, 224, 224])
Number of images per class in the train set:
[(0, 1000), (1, 1000), (2, 1000), (3, 1000), (4, 1000), (5, 1000), (6, 1000), (7, 1000), (8, 1000), (9, 1000)]
The test set contains 10000 images, in 312 batches


In [20]:
my_resnet = models.resnet18(num_classes=10)
my_resnet.conv1 = torch.nn.Conv2d(num_channels, my_resnet.conv1.out_channels, kernel_size=my_resnet.conv1.kernel_size, stride=my_resnet.conv1.stride, padding=my_resnet.conv1.padding, bias=my_resnet.conv1.bias)
my_resnet.load_state_dict(torch.load(model_weigths_path + "resnet18_fine_tuned2_MNIST.pth"))

C:\Users\bocca\AppData\Local\Temp\ipykernel_18364\2653926941.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  my_resnet.load_state_dict(torch.load(model_weigths_path + "r

<All keys matched successfully>

In [18]:
def compute_classif_metrics(y_test,y_test_predictions):
    accuracy = accuracy_score(y_test, y_test_predictions)
    precision = precision_score(y_test, y_test_predictions, average = 'macro', zero_division=np.nan)
    recall = recall_score(y_test, y_test_predictions, average = 'macro', zero_division=np.nan)
    f1score = f1_score(y_test, y_test_predictions, average = 'macro', zero_division=np.nan)
    return accuracy, precision, recall, f1score

In [21]:
loss_function = torch.nn.CrossEntropyLoss() # cross entropy works well for multi-class problems
# releasing unceseccary memory in GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
nb_test_batches = len(test_loader)

with torch.no_grad():
    my_resnet.eval()
    total_test_loss = 0
    precision, recall, f1score, accuracy = [], [], [], []
    for (input, target) in test_loader:
        input.to(device)
        target.to(device)
        output = my_resnet(input)
        loss = loss_function(output, target)
        total_test_loss += loss.item()
        predicted_classes = torch.max(output, 1)[1] # get class from network's prediction
            
        # calculate P/R/F1/A metrics for batch
        acc, prec, rec, f1s = compute_classif_metrics(target.cpu(), predicted_classes.cpu())
        accuracy.append(acc)
        precision.append(prec)
        recall.append(rec)
        f1score.append(f1s)
        
    print(f"Test loss: {total_test_loss/nb_test_batches}")
    print(f"Accuracy =  %.4f, Precision =  %.4f, Recall =  %.4f, F1Score =  %.4f" % (np.sum(accuracy)/nb_test_batches, np.sum(precision)/nb_test_batches, np.sum(recall)/nb_test_batches, np.sum(f1score)/nb_test_batches))
        

Test loss: 0.6394441552364674
Accuracy =  0.7517, Precision =  0.7960, Recall =  0.7528, F1Score =  0.7050
